In [1]:
# 🚀 FULL ONE-CELL SETUP + TRAINING + TESTING
!pip uninstall -y transformers datasets fsspec gcsfs accelerate > /dev/null
!pip install -q transformers==4.45.2 datasets==2.18.0 torch fsspec==2024.3.1 gcsfs==2024.3.1 accelerate

# ---- IMPORTS ----
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

print("✅ Transformers:", __import__('transformers').__version__)
print("✅ GPU available:", torch.cuda.is_available())

# ---- LOAD DATASET ----
dataset = load_dataset("imdb")
train_dataset = dataset["train"].shuffle(seed=42).select(range(3000))
test_dataset  = dataset["test"].shuffle(seed=42).select(range(500))

# ---- TOKENIZE ----
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test  = test_dataset.map(tokenize_fn, batched=True)
tokenized_train.set_format("torch", columns=["input_ids","attention_mask","label"])
tokenized_test.set_format("torch",  columns=["input_ids","attention_mask","label"])

# ---- LOAD MODEL ----
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# ---- TRAINING CONFIG ----
training_args = TrainingArguments(
    output_dir="./bert-sentiment",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
)

# ---- TRAINER ----
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

# ---- TRAIN ----
print("🚀 Training started...")
trainer.train()
print("✅ Training completed!")

# ---- EVALUATION ----
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

preds = trainer.predict(tokenized_test)
metrics = compute_metrics(preds)
print("\n📊 Evaluation Results:")
for k,v in metrics.items():
    print(f"{k}: {v:.4f}")

# ---- SAVE MODEL ----
model.save_pretrained("./sentiment-bert-model")
tokenizer.save_pretrained("./sentiment-bert-tokenizer")

# ---- TEST CUSTOM SENTENCES ----
sentiment_analyzer = pipeline("sentiment-analysis", model="./sentiment-bert-model", tokenizer="./sentiment-bert-tokenizer")
texts = [
    "The movie was absolutely amazing and emotional!",
    "I hated the plot, it was boring and predictable.",
    "Such a fantastic performance by the actors!",
    "Not my type of movie, very slow pacing."
]
print("\n💬 Custom Sentence Predictions:")
for t in texts:
    print(f"{t} → {sentiment_analyzer(t)}")

# ---- SUMMARY ----
print("""
📘 **Sentiment Analysis using BERT**
-----------------------------------
• Model: bert-base-uncased fine-tuned on IMDb
• Dataset: 3000 train, 500 test
• Epochs: 1 (for quick run)
• Metrics: Accuracy, Precision, Recall, F1
• Output: Fine-tuned model saved in ./sentiment-bert-model
• Custom sentences classified successfully ✅
""")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.4 MB/s eta 0:00:00
ERROR: Cannot install datasets==2.18.0 and fsspec==2024.3.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


ModuleNotFoundError: No module named 'datasets'

In [2]:
# 🚀 Clean install of only what's needed (no version pinning)
!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets accelerate scikit-learn

# ---- IMPORTS ----
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

print("✅ Transformers:", __import__('transformers').__version__)
print("✅ Datasets:", __import__('datasets').__version__)
print("✅ GPU available:", torch.cuda.is_available())

# ---- LOAD DATASET ----
dataset = load_dataset("imdb")
train_dataset = dataset["train"].shuffle(seed=42).select(range(3000))
test_dataset  = dataset["test"].shuffle(seed=42).select(range(500))

# ---- TOKENIZE ----
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test  = test_dataset.map(tokenize_fn, batched=True)
tokenized_train.set_format("torch", columns=["input_ids","attention_mask","label"])
tokenized_test.set_format("torch",  columns=["input_ids","attention_mask","label"])

# ---- LOAD MODEL ----
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# ---- TRAINING CONFIG ----
training_args = TrainingArguments(
    output_dir="./bert-sentiment",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)

# ---- TRAINER ----
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

# ---- TRAIN ----
print("🚀 Training started...")
trainer.train()
print("✅ Training completed!")

# ---- EVALUATE ----
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

preds = trainer.predict(tokenized_test)
metrics = compute_metrics(preds)
print("\n📊 Evaluation Results:")
for k,v in metrics.items():
    print(f"{k}: {v:.4f}")

# ---- TEST CUSTOM SENTENCES ----
sentiment_analyzer = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
texts = [
    "The movie was absolutely amazing and emotional!",
    "I hated the plot, it was boring and predictable.",
    "Such a fantastic performance by the actors!",
    "Not my type of movie, very slow pacing."
]
print("\n💬 Custom Sentence Predictions:")
for t in texts:
    print(f"{t} → {sentiment_analyzer(t)}")

# ---- SUMMARY ----
print("""
📘 **Sentiment Analysis using BERT**
-----------------------------------
• Model: bert-base-uncased fine-tuned on IMDb
• Dataset: 3000 train, 500 test
• Epochs: 1 (for quick run)
• Metrics: Accuracy, Precision, Recall, F1
• Output: Fine-tuned model trained and evaluated successfully ✅
""")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires gcsfs!=2025.5.0,>=2023.3.0, which is not installed.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, which is not installed.
peft 0.17.1 requires accelerate>=0.21.0, which is not installed.
peft 0.17.1 requires transformers, which is not installed.
torchtune 0.6.1 requires datasets, which is not installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 13.2 MB/s eta 0:

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [2]:
import os
os.environ["WANDB_MODE"] = "disabled"


In [3]:
# --- Single-cell: train BERT (works even if TrainingArguments lacks evaluation_strategy) ---
# (Run this in your existing Colab session — no pip installs here)

from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

print("Python and torch info:")
print("  torch.cuda.is_available():", torch.cuda.is_available())

# 1) Load a small subset of IMDb to keep runtime light
dataset = load_dataset("imdb")
train_dataset = dataset["train"].shuffle(seed=42).select(range(3000))
test_dataset  = dataset["test"].shuffle(seed=42).select(range(500))

# 2) Tokenize
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test  = test_dataset.map(tokenize_fn, batched=True)
tokenized_train.set_format("torch", columns=["input_ids","attention_mask","label"])
tokenized_test.set_format("torch",  columns=["input_ids","attention_mask","label"])

# 3) Load model (warning about classifier.* is normal — it's newly initialized)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# 4) Minimal TrainingArguments (compatible with older/newer variants)
training_args = TrainingArguments(
    output_dir="./bert-sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=4,   # reduce to 2 if you hit OOM
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
)

# 5) Trainer setup & train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=None  # we'll run evaluation manually below to avoid relying on eval hooks
)

print("Starting training... (this may take a few minutes)")
trainer.train()
print("Training finished.")

# 6) Evaluate manually using trainer.predict on the test set
print("Running prediction on test set...")
predictions = trainer.predict(tokenized_test)  # returns (preds, labels, metrics)
logits = predictions.predictions
labels = predictions.label_ids
preds = logits.argmax(-1)

precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
acc = accuracy_score(labels, preds)
print("\nEvaluation metrics (manual):")
print(f"  accuracy:  {acc:.4f}")
print(f"  precision: {precision:.4f}")
print(f"  recall:    {recall:.4f}")
print(f"  f1:        {f1:.4f}")

# 7) Save model & tokenizer
model.save_pretrained("./sentiment-bert-model")
tokenizer.save_pretrained("./sentiment-bert-tokenizer")
print("\nSaved model and tokenizer to './sentiment-bert-model' and './sentiment-bert-tokenizer'")

# 8) Quick custom-sentence tests
sentiment_pipe = pipeline("sentiment-analysis", model="./sentiment-bert-model", tokenizer="./sentiment-bert-tokenizer")
examples = [
    "The movie was absolutely amazing and emotional!",
    "I hated the plot, it was boring and predictable.",
    "The acting was great but the story was weak.",
    "Terrible. I walked out halfway through."
]
print("\nCustom sentence predictions:")
for s in examples:
    print(f"  {s} -> {sentiment_pipe(s)}")


Python and torch info:
  torch.cuda.is_available(): False


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting training... (this may take a few minutes)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.681600
100,0.505600
150,0.606700
200,0.471200
250,0.489400
300,0.427600
350,0.406300


...

Training finished.
Running prediction on test set...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Evaluation metrics (manual):
  accuracy:  0.8540
  precision: 0.8145
  recall:    0.9106
  f1:        0.8599

Saved model and tokenizer to './sentiment-bert-model' and './sentiment-bert-tokenizer'


Device set to use cpu



Custom sentence predictions:
  The movie was absolutely amazing and emotional! -> [{'label': 'LABEL_1', 'score': 0.9928454756736755}]
  I hated the plot, it was boring and predictable. -> [{'label': 'LABEL_0', 'score': 0.9884673953056335}]
  The acting was great but the story was weak. -> [{'label': 'LABEL_0', 'score': 0.9536903500556946}]
  Terrible. I walked out halfway through. -> [{'label': 'LABEL_0', 'score': 0.9815766215324402}]
